In [63]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
import statsmodels.api as sm
from linearmodels import PooledOLS, PanelOLS
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from linearmodels.panel import RandomEffects
#from sklearn.feature_selection import chi2
from scipy.stats import chi2 as scipy_chi2
import statsmodels.stats.stattools as sm_stattools

In [29]:
wb_panel = pd.read_csv('world_bank_panel_data.csv')
wb_panel.head()

,country,year,inflation_rate,gdp_per_capita,gov_health_exp,oop_spending,pop_65_plus,log_gdp_per_capita
0,ABW,2000,4.044021,20681.023027,NaN,NaN,6.778277,9.937020
1,ABW,2001,2.883604,20740.132583,NaN,NaN,7.016223,9.939874
2,ABW,2002,3.315247,21307.248251,NaN,NaN,7.282895,9.966850
3,ABW,2003,3.656365,21949.485996,NaN,NaN,7.551728,9.996545
4,ABW,2004,2.529129,23700.631990,NaN,NaN,7.842296,10.073299


In [30]:
wb_panel.info()

<class 'pandas.DataFrame'>
RangeIndex: 4700 entries, 0 to 4699
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             4700 non-null   str    
 1   year                4700 non-null   int64  
 2   inflation_rate      4526 non-null   float64
 3   gdp_per_capita      4661 non-null   float64
 4   gov_health_exp      4351 non-null   float64
 5   oop_spending        4366 non-null   float64
 6   pop_65_plus         4700 non-null   float64
 7   log_gdp_per_capita  4661 non-null   float64
dtypes: float64(6), int64(1), str(1)
memory usage: 307.6 KB


In [31]:
wb_panel.isnull().mean() * 100

country               0.000000
year                  0.000000
inflation_rate        3.702128
gdp_per_capita        0.829787
gov_health_exp        7.425532
oop_spending          7.106383
pop_65_plus           0.000000
log_gdp_per_capita    0.829787
dtype: float64

In [32]:
# Calculate missing values by year for inflation_rate
missing_by_year = wb_panel.groupby(['year', 'country'])['inflation_rate'].apply(lambda x: x.isna().mean()*100).sort_index()

In [33]:
# Display the results
print("Missing values for inflation_rate by year:")
print(missing_by_year)

Missing values for inflation_rate by year:
year  country
2000  ABW          0.0
      AFE          0.0
      AFG        100.0
      AFW          0.0
      AGO          0.0
                 ...  
2024  XKX          0.0
      YEM        100.0
      ZAF          0.0
      ZMB          0.0
      ZWE        100.0
Name: inflation_rate, Length: 4700, dtype: float64


In [70]:
# Check if the missing data is tied to specific YEARS (e.g., no data before 2005)
print("--- Percentage of Missing Inflation Data by YEAR ---")
missing_by_year = wb_panel.groupby('year')['inflation_rate'].apply(lambda x: x.isnull().mean() * 100)
print(missing_by_year.round(1).astype(str) + '%')

# Check if the missing data is tied to specific COUNTRIES 
print("\n--- Countries Missing 100% of Inflation Data ---")
missing_by_country = wb_panel.groupby('country')['inflation_rate'].apply(lambda x: x.isnull().mean() * 100)
completely_missing_countries = missing_by_country[missing_by_country == 100.0]

if completely_missing_countries.empty:
    print("No country is missing 100% of its data.")
else:
    print(completely_missing_countries.astype(str) + '%')
    print(f"\nTotal countries with ZERO inflation data: {len(completely_missing_countries)}")

--- Percentage of Missing Inflation Data by YEAR ---
year
2000    12.2%
2001     9.6%
2002     8.5%
2003     6.9%
2004     6.4%
2005     4.8%
2006     3.7%
2007     2.7%
2008     2.1%
2009     1.1%
2010     0.5%
2011     0.0%
2012     0.0%
2013     0.0%
2014     0.0%
2015     0.5%
2016     0.5%
2017     1.6%
2018     2.1%
2019     2.1%
2020     4.8%
2021     4.8%
2022     4.8%
2023     5.9%
2024     6.9%
Name: inflation_rate, dtype: str

--- Countries Missing 100% of Inflation Data ---
No country is missing 100% of its data.


In [35]:
# list comprehension for countries will 100% null values for inflation_rate
completely_missing_countries = [country for country in missing_by_country.index if missing_by_country[country] == 100.0]
print(f"\nCountries with 100% missing inflation data: {completely_missing_countries}")
print(f'\nTotal countries :{wb_panel["country"].nunique()}')
print(f'{len(completely_missing_countries)/wb_panel["country"].nunique() * 100:.1f}% of countries have 100% missing inflation data')


Countries with 100% missing inflation data: []

Total countries :188
0.0% of countries have 100% missing inflation data


In [ ]:
# 1. Define the columns that need patching
features_to_impute = [
    'inflation_rate', 'gdp_per_capita', 'gov_health_exp', 
    'oop_spending', 'pop_65_plus', 'log_gdp_per_capita'
]

# Initialize the imputer (looking at the 5 most similar countries to fill gaps)
imputer = KNNImputer(n_neighbors=5, weights='distance')

# Apply the imputer to our clean, sovereign-only dataset
wb_final = wb_panel.copy()
wb_final[features_to_impute] = imputer.fit_transform(wb_final[features_to_impute])

print("--- FINAL MISSING DATA CHECK ---")
print(wb_final.isnull().sum())

--- FINAL MISSING DATA CHECK ---
country               0
year                  0
inflation_rate        0
gdp_per_capita        0
gov_health_exp        0
oop_spending          0
pop_65_plus           0
log_gdp_per_capita    0
dtype: int64


In [41]:
# correlation matrix for the imputed dataset
corr_matrix = wb_final[features_to_impute].corr()
print("\n--- Correlation Matrix for Imputed Dataset ---")
print(corr_matrix.round(2))   


--- Correlation Matrix for Imputed Dataset ---
                    inflation_rate  gdp_per_capita  gov_health_exp  \
inflation_rate                1.00           -0.13           -0.13   
gdp_per_capita               -0.13            1.00            0.52   
gov_health_exp               -0.13            0.52            1.00   
oop_spending                  0.09           -0.38           -0.63   
pop_65_plus                  -0.15            0.57            0.69   
log_gdp_per_capita           -0.20            0.80            0.61   

                    oop_spending  pop_65_plus  log_gdp_per_capita  
inflation_rate              0.09        -0.15               -0.20  
gdp_per_capita             -0.38         0.57                0.80  
gov_health_exp             -0.63         0.69                0.61  
oop_spending                1.00        -0.31               -0.44  
pop_65_plus                -0.31         1.00                0.68  
log_gdp_per_capita         -0.44         0.68        

In [ ]:
# set multi index for panel regression
wb_final_panel = wb_final.copy()

wb_final_panel.set_index(['country', 'year'], inplace=True)

# seperate response variable for regression
response = wb_final_panel['oop_spending']

# seperate regressors for regression
regressors = wb_final_panel[['inflation_rate', 'gov_health_exp', 'pop_65_plus', 'log_gdp_per_capita']]

# add intercept for regression
regressors = sm.add_constant(regressors)

mod_pooled = PooledOLS(response, regressors)
res_pooled = mod_pooled.fit(cov_type='clustered', cluster_entity=True)
print(res_pooled.summary)


                          PooledOLS Estimation Summary                          
Dep. Variable:           oop_spending   R-squared:                        0.4582
Estimator:                  PooledOLS   R-squared (Between):              0.5192
No. Observations:                4700   R-squared (Within):               0.1953
Date:                Thu, May 21 2026   R-squared (Overall):              0.4582
Time:                        23:54:01   Log-likelihood                -1.917e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      992.61
Entities:                         188   P-value                           0.0000
Avg Obs:                       25.000   Distribution:                  F(4,4695)
Min Obs:                       25.000                                           
Max Obs:                       25.000   F-statistic (robust):             78.218
                            

In [ ]:
# check for multicollinearity using Variance Inflation Factor (VIF)
vif_data = pd.DataFrame()
vif_data['feature'] = regressors.columns
vif_data['VIF'] = [variance_inflation_factor(regressors.values, i) for i in range(regressors.shape[1])]
print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data.round(2))


--- Variance Inflation Factor (VIF) ---
              feature    VIF
0               const  46.02
1      inflation_rate   1.04
2      gov_health_exp   2.05
3         pop_65_plus   2.40
4  log_gdp_per_capita   2.05


In [52]:
# Check for heteroscadacity using Breusch-Pagan test
bp_test = het_breuschpagan(res_pooled.resids, regressors)
print(f'p-value for Breusch-Pagan test: {bp_test[1]:.4f}')

p-value for Breusch-Pagan test: 0.0000


In [64]:
# check for serial correlation using Durbin-Watson test
dw_stat = sm_stattools.durbin_watson(res_pooled.resids)
print(f'Durbin-Watson statistic: {dw_stat:.2f}')

Durbin-Watson statistic: 0.29


In [67]:
# Hausman test for fixed effects vs random effects

re_model = RandomEffects(response, regressors).fit()
fe_model = PanelOLS(response, regressors, entity_effects=True).fit(cov_type='clustered', cluster_entity=True)

print("\n--- Hausman Test (FE vs RE) ---")
diff     = fe_model.params - re_model.params
diff_cov = fe_model.cov - re_model.cov
stat     = diff @ np.linalg.inv(diff_cov) @ diff
pval     = 1 - scipy_chi2(len(diff)).cdf(stat)
print(f"Hausman Statistic: {stat.round(4)}")
print(f"P-Value: {pval.round(4)}")



--- Hausman Test (FE vs RE) ---
Hausman Statistic: 0.5083
P-Value: 0.9918


In [68]:
# print the summary of the fixed effects model
print("\n--- Fixed Effects Model Summary ---")
print(fe_model.summary)



--- Fixed Effects Model Summary ---
                          PanelOLS Estimation Summary                           
Dep. Variable:           oop_spending   R-squared:                        0.2388
Estimator:                   PanelOLS   R-squared (Between):              0.4586
No. Observations:                4700   R-squared (Within):               0.2388
Date:                Fri, May 22 2026   R-squared (Overall):              0.4172
Time:                        19:42:03   Log-likelihood                -1.605e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      353.62
Entities:                         188   P-value                           0.0000
Avg Obs:                       25.000   Distribution:                  F(4,4508)
Min Obs:                       25.000                                           
Max Obs:                       25.000   F-statistic (robust):           

In [69]:
# check for heteroscadacity and serial correlation in the fixed effects model
bp_test_fe = het_breuschpagan(fe_model.resids, regressors)
dw_stat = sm_stattools.durbin_watson(fe_model.resids)
print(f'Breusch-Pagan test statistic: {bp_test_fe[0]:.2f}')
print(f'p-value: {bp_test_fe[1]:.2f}')
print(f'Durbin-Watson statistic: {dw_stat:.2f}')

Breusch-Pagan test statistic: 184.38
p-value: 0.00
Durbin-Watson statistic: 1.12
